# DEG20 Downstream Experiment

多条件版 scPRAM-style DEG20 downstream 实验。这个 notebook 直接消费现有 full-gene `pkl`，不重新训练模型。

核心定义：
- 真实 DEG20：直接使用 `DE_idx / DE_name`，即我们当前 TriShift 口径的 `top20_degs_non_dropout`。
- 预测 DEG20：从 `Pred_full / Ctrl_full` 生成。多行预测优先走 Scanpy 排名；单行预测自动退化为 effect-size 排名。
- 主指标：`common_degs_at_20`、`jaccard_at_20`、`precision_at_20`、`recall_at_20`，以及限定在 truth DEG20 上的 `scpram_r2_degs_mean_mean / scpram_r2_degs_var_mean / scpram_wasserstein_degs_sum`。

默认推荐：
- `pred_deg_mode = "adaptive"`
- `enrichment_mode = "export_only"`
- `variant_tag = "nearest"` 仅在 `model_name="trishift"` 且目标产物是 `trishift_*_nearest.pkl` 时需要设置


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / "scripts").exists() and (repo_root.parent / "scripts").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import scripts.trishift.analysis.deg20_experiment as deg20_experiment
importlib.reload(deg20_experiment)

run_deg20_experiment = deg20_experiment.run_deg20_experiment
load_condition_payload = deg20_experiment.load_condition_payload
summarize_condition_payload = deg20_experiment.summarize_condition_payload
build_mean_var_scatter = deg20_experiment.build_mean_var_scatter

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
repo_root


## Parameters

设置一个模型、一个数据集，以及要分析的 split。


In [ ]:
dataset = "adamson"
model_name = "trishift"  # trishift | gears | genepert | scouter
split_ids = [1, 2, 3, 4, 5]
result_dir = None  # 可手动覆盖结果目录
out_root = None  # 可手动覆盖 notebook 输出目录
variant_tag = None  # trishift 常用: None / nearest / random
focus_conditions = []  # 例如 ["ATP5B+ctrl", "BHLHE40+ctrl"]
pred_deg_mode = "adaptive"  # adaptive | scanpy | effect_size
enrichment_mode = "export_only"  # export_only | run_if_available | disabled
enrichment_library = "Reactome_2022"
remove_perturbed_genes = True

run_kwargs = dict(
    dataset=dataset,
    model_name=model_name,
    split_ids=split_ids,
    result_dir=result_dir,
    out_root=out_root,
    variant_tag=variant_tag,
    focus_conditions=focus_conditions or None,
    pred_deg_mode=pred_deg_mode,
    enrichment_mode=enrichment_mode,
    enrichment_library=enrichment_library,
    remove_perturbed_genes=remove_perturbed_genes,
)
run_kwargs


In [ ]:
result = run_deg20_experiment(**run_kwargs)
print(f"out_dir: {result.out_dir}")
display(result.dataset_summary_df)
display(result.split_summary_df)
display(result.per_condition_df.head(20))


## Metric Distributions

这里先看每个 condition 的 overlap / regression / Wasserstein 指标分布。


In [ ]:
metric_cols = [
    "common_degs_at_20",
    "jaccard_at_20",
    "precision_at_20",
    "recall_at_20",
    "scpram_r2_degs_mean_mean",
    "scpram_r2_degs_var_mean",
    "scpram_wasserstein_degs_sum",
]
plot_df = result.per_condition_df[metric_cols].apply(pd.to_numeric, errors="coerce")
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
axes = axes.ravel()
for ax, col in zip(axes, metric_cols):
    vals = plot_df[col].dropna()
    ax.hist(vals, bins=min(20, max(5, len(vals))))
    ax.set_title(col)
for ax in axes[len(metric_cols):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

display(result.per_condition_df[metric_cols].describe().T)


## Representative Conditions

默认会自动挑 overlap 最好 / 中位 / 最差的 3 个 condition；如果你在参数里填了 `focus_conditions`，这里就按你指定的 condition 展示。


In [ ]:
rep_cols = [
    "split_id",
    "condition",
    "pred_deg_mode_used",
    "common_degs_at_20",
    "jaccard_at_20",
    "scpram_r2_degs_mean_mean",
    "scpram_r2_degs_var_mean",
    "scpram_wasserstein_degs_sum",
]
display(result.representative_df[rep_cols])


In [ ]:
for row in result.representative_df.to_dict(orient="records"):
    split_id = int(row["split_id"])
    condition = str(row["condition"])
    payload_item = load_condition_payload(
        model_name=model_name,
        dataset=dataset,
        split_id=split_id,
        condition=condition,
        result_dir=result_dir,
        variant_tag=variant_tag,
    )
    detail = summarize_condition_payload(
        payload_item=payload_item,
        condition=condition,
        pred_deg_mode=pred_deg_mode,
        remove_perturbed_genes=remove_perturbed_genes,
    )
    print(f"\n=== split {split_id} | {condition} ===")
    display(
        pd.DataFrame(
            {
                "truth_deg20": pd.Series(detail["truth_deg20"]),
                "pred_deg20": pd.Series(detail["pred_deg20"]),
                "common_deg20": pd.Series(detail["common_deg20"]),
            }
        )
    )
    fig_mean, fig_var = build_mean_var_scatter(
        payload_item=payload_item,
        truth_deg_idx=detail["truth_deg_idx"],
        title_prefix=f"split {split_id} | {condition}",
    )
    plt.show(fig_mean)
    plt.show(fig_var)
    plt.close(fig_mean)
    plt.close(fig_var)


## Exported Gene Lists

主流程默认总会导出三类 gene list：`truth_deg20` / `pred_deg20` / `common_deg20`。


In [ ]:
display(result.gene_lists_df.head(30))
print(result.out_dir / "deg_gene_lists_long.csv")


## Optional Enrichment

当 `enrichment_mode="run_if_available"` 且环境里有 `gseapy` 时，这里会显示 enrichment 结果；否则 notebook 只保留 gene list 导出。


In [ ]:
if result.enrichment_df.empty:
    print("No enrichment results. This is expected when enrichment_mode='export_only' or gseapy is unavailable.")
else:
    display(result.enrichment_df.head(30))


## Notes

- `common_degs_at_20` 和 `jaccard_at_20` 更直接回答“预测 DEG20 和 truth DEG20 重合多少”。
- `scpram_r2_degs_mean_mean` / `scpram_r2_degs_var_mean` 更接近 scPRAM 论文里 DEG mean/variance regression 的思想。
- `scpram_wasserstein_degs_sum` 越低越好，表示预测和真实在 DEG20 子空间上的分布差更小。
